# 3. Анализ и оформление результатов

## 3.1. Импорты и настройка

In [1]:
import json, os, subprocess, sys
import pandas as pd
import matplotlib.pyplot as plt

os.makedirs('data/processed', exist_ok=True)
plt.style.use('seaborn-v0_8-darkgrid')

## 3.2. Загрузка метрик и создание сводной таблицы

In [2]:
# Загружаем метрики
with open('data/processed/all_models_metrics.json', 'r') as f:
    metrics = json.load(f)

# Создаём DataFrame
df_metrics = pd.DataFrame(metrics).T    # транспонируем, чтобы модели были строками
df_metrics.index.name = 'Model'

# Округляем для читаемости
df_metrics = df_metrics.round(4)

print("Сводная таблица метрик:")
display(df_metrics)

# Сохраняем в CSV
df_metrics.to_csv('data/processed/comparison_table.csv')

# Находим лучшую модель по R²
best_model = df_metrics['R2'].idxmax()
best_r2 = df_metrics.loc[best_model, 'R2']
print(f"\nЛучшая модель по R²: **{best_model}** (R² = {best_r2})")

Сводная таблица метрик:


,MSE,MAE,R2
Model,,,
linear_regression,0.2284,0.2867,0.7237
decision_tree,0.1712,0.2309,0.7928
catboost,0.1707,0.2553,0.7935
xgboost,0.1711,0.2133,0.7929
neural_network,0.1756,0.2445,0.7875



Лучшая модель по R²: **catboost** (R² = 0.7935)


## 3.3. Генерация графа DVC

In [3]:
# Попробуем сгенерировать граф DVC с помощью встроенной команды
dag_path = 'data/processed/dag.png'
try:
    # Генерируем DOT-описание и передаём в Graphviz (требуется установленный Graphviz)
    dot = subprocess.check_output([sys.executable, '-m', 'dvc', 'dag', '--dot'], 
                                  text=True, stderr=subprocess.STDOUT)
    # Сохраняем в PNG через graphviz Python-библиотеку
    from graphviz import Source
    s = Source(dot, format='png')
    s.render(filename='dag', directory='data/processed', cleanup=True)
    print(f"Граф DVC сохранён в {dag_path}")
except Exception as e:
    print(f"Не удалось построить изображение графа ({e}).")
    print("Вы можете выполнить команду `dvc dag` в терминале и прикрепить скриншот.")

Граф DVC сохранён в data/processed/dag.png
